In [1]:
"""
============================================================
ORZYN AI m2.0
Notebook 04
Commit Intelligence
============================================================

Purpose
-------
Extract and analyze commit history for any GitHub repository.

Produces
--------
CommitProfile objects
Repository commit metrics
Developer activity metrics
Timeline statistics
"""

'\n============================================================\nORZYN AI m2.0\nNotebook 04\nCommit Intelligence\n============================================================\n\nPurpose\n-------\nExtract and analyze commit history for any GitHub repository.\n\nProduces\n--------\nCommitProfile objects\nRepository commit metrics\nDeveloper activity metrics\nTimeline statistics\n'

In [2]:
from pathlib import Path 
import sys 

BACKEND_DIR = Path.cwd().parent.resolve() 
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

In [3]:
from collections import Counter
from dataclasses import dataclass
from datetime import datetime

from orzyn import *

In [4]:
repo_config = get_active_repository()

OWNER = repo_config.owner

REPOSITORY = repo_config.repository

In [5]:
COMMIT_HISTORY_QUERY = """
query(
    $owner:String!,
    $name:String!
){

repository(
    owner:$owner,
    name:$name
){

defaultBranchRef{

target{

... on Commit{

history(first:100){

nodes{

oid

messageHeadline

committedDate

additions

deletions

changedFilesIfAvailable

author{

name

email

user{

login

}

}

}

}

}

}

}

}

}
"""

In [6]:
@dataclass(slots=True)
class CommitProfile:

    sha: str

    message: str

    author: str

    username: str | None

    email: str | None

    committed_at: datetime

    additions: int

    deletions: int

    files_changed: int

In [7]:
def build_commit(node: dict) -> CommitProfile:

    author = node.get("author") or {}

    user = author.get("user")

    return CommitProfile(

        sha=node["oid"],

        message=node["messageHeadline"],

        author=author.get("name") or "Unknown",

        username=user.get("login") if user else None,

        email=author.get("email"),

        committed_at=parse_datetime(
            node["committedDate"]
        ),

        additions=node.get("additions", 0),

        deletions=node.get("deletions", 0),

        files_changed=node.get(
            "changedFilesIfAvailable"
        ) or 0

    )

In [8]:
repository = client.execute(

    """
    query(
        $owner:String!,
        $name:String!
    ){

        repository(
            owner:$owner,
            name:$name
        ){

            name

            defaultBranchRef{

                name

                target{

                    __typename

                }

            }

        }

    }
    """,

    {

        "owner": OWNER,

        "name": REPOSITORY

    }

)

In [9]:
repo = repository.get("repository")

if repo is None:

    raise ValueError(
        "Repository not found."
    )

branch = repo.get("defaultBranchRef")

if branch is None:

    raise ValueError(
        "Repository has no default branch."
    )

target = branch.get("target")

if target is None:

    raise ValueError(
        "Default branch contains no commits."
    )

if target["__typename"] != "Commit":

    raise ValueError(
        "Default branch target is not a Commit."
    )

print(f"Repository : {repo['name']}")
print(f"Branch     : {branch['name']}")

Repository : kennykrichardson-portfolio
Branch     : main


In [10]:
result = client.execute(

    COMMIT_HISTORY_QUERY,

    {
        "owner": OWNER,
        "name": REPOSITORY
    }

)

repository = result.get("repository")

if repository is None:
    raise RuntimeError("Repository not found.")

branch = repository.get("defaultBranchRef")

if branch is None:
    raise RuntimeError("Repository has no default branch.")

target = branch.get("target")

if target is None:
    raise RuntimeError("Default branch has no commits.")

history = target.get("history")

if history is None:
    raise RuntimeError("Unable to retrieve commit history.")

nodes = history.get("nodes", [])

commits = [
    build_commit(node)
    for node in nodes
]

print(f"Downloaded {len(commits)} commits.")

Downloaded 15 commits.


In [11]:
if not commits:

    raise RuntimeError(
        "Repository contains no commits."
    )

print("Commit history validated.")

Commit history validated.


In [12]:
commits[0]

CommitProfile(sha='c3781ac6d07358aed915bb032b804cefc34d7421', message='Portfolio Mark III', author='Ken Richardson', username='kennykrichardson', email='kennykrichardson@gmail.com', committed_at=datetime.datetime(2026, 7, 11, 12, 5, 28, tzinfo=datetime.timezone.utc), additions=101, deletions=13, files_changed=6)

In [13]:
commits[-1]

CommitProfile(sha='6eba744eefbc06b8801048dd3ef86696ba881e38', message='Portfolio Mark I', author='Ken Richardson', username='kennykrichardson', email='kennykrichardson@gmail.com', committed_at=datetime.datetime(2026, 7, 1, 9, 31, 12, tzinfo=datetime.timezone.utc), additions=3407, deletions=0, files_changed=47)

In [14]:
author_counts = Counter(

    commit.author

    for commit in commits

)

author_counts

Counter({'Ken Richardson': 14, 'Kenny Richardson Kodipally': 1})

In [15]:
author_counts.most_common(10)

[('Ken Richardson', 14), ('Kenny Richardson Kodipally', 1)]

In [16]:
sorted_commits = sorted(

    commits,

    key=lambda commit: (

        commit.additions +

        commit.deletions

    ),

    reverse=True

)

largest_commit = sorted_commits[0]

print(

    f"Ranked {len(sorted_commits)} commits "

    "by total lines changed."

)

Ranked 15 commits by total lines changed.


In [17]:
print("=" * 120)

print(
    f"{'#':<4}"
    f"{'Message':<65}"
    f"{'+Adds':>8}"
    f"{'-Dels':>8}"
    f"{'Total':>10}"
)

print("=" * 120)

for index, commit in enumerate(

    sorted_commits[:100],

    start=1

):

    total = (

        commit.additions +

        commit.deletions

    )

    print(

        f"{index:<4}"

        f"{commit.message[:62]:<65}"

        f"{commit.additions:>8}"

        f"{commit.deletions:>8}"

        f"{total:>10}"

    )

#   Message                                                             +Adds   -Dels     Total
1   Portfolio Mark I                                                     3407       0      3407
2   Portfolio Mark I                                                       30     219       249
3   Portfolio Mark I                                                      127      77       204
4   Updated Portfolio                                                     115      19       134
5   Portfolio Mark III                                                    101      13       114
6   Portfolio Mark III                                                     48      20        68
7   Portfolio Mark II                                                      21       3        24
8   Portfolio Mark I                                                       17       0        17
9   Portfolio Mark I                                                        2       2         4
10  Merge branch 'main' of https://githu

In [18]:
average_changes = (

    sum(

        commit.additions +

        commit.deletions

        for commit in commits

    )

    / len(commits)

)

round(

    average_changes,

    2

)

282.07

In [19]:
weekday_counts = Counter(

    commit.committed_at.strftime("%A")

    for commit in commits

)

weekday_counts

Counter({'Friday': 8,
         'Saturday': 2,
         'Tuesday': 2,
         'Thursday': 2,
         'Wednesday': 1})

In [20]:
hour_counts = Counter(

    commit.committed_at.hour

    for commit in commits

)

hour_counts

Counter({7: 4, 12: 3, 5: 2, 13: 1, 10: 1, 8: 1, 6: 1, 3: 1, 9: 1})

In [21]:
first_commit = min(

    commits,

    key=lambda commit:

        commit.committed_at

)

latest_commit = max(

    commits,

    key=lambda commit:

        commit.committed_at

)

print(first_commit.committed_at)

print(latest_commit.committed_at)

2026-07-01 09:31:12+00:00
2026-07-11 12:05:28+00:00


In [22]:
print("=" * 60)

print("Repository Commit Summary")

print("=" * 60)

print()

print(f"Repository      : {REPOSITORY}")

print(f"Total Commits   : {len(commits)}")

print(f"Contributors    : {len(author_counts)}")

print(
    f"Largest Commit  : "
    f"{largest_commit.message}"
)

print(
    f"Lines Changed   : "
    f"{largest_commit.additions + largest_commit.deletions}"
)

print(
    f"Author          : "
    f"{largest_commit.author}"
)
print(
    f"Average Change  : "
    f"{average_changes:.2f}"
)

print(
    f"First Commit    : "
    f"{first_commit.committed_at.date()}"
)

print(
    f"Latest Commit   : "
    f"{latest_commit.committed_at.date()}"
)

Repository Commit Summary

Repository      : kennykrichardson-portfolio
Total Commits   : 15
Contributors    : 2
Largest Commit  : Portfolio Mark I
Lines Changed   : 3407
Author          : Ken Richardson
Average Change  : 282.07
First Commit    : 2026-07-01
Latest Commit   : 2026-07-11
